# SupportOps AI - BANKING77 Dataset Validation

## Objective

Evaluate BANKING77 as the primary intent-classification dataset for
SupportOps AI before training classical ML and Transformer models.

The validation process checks:

- Dataset structure
- Intent labels
- Missing values
- Exact duplicates
- Conflicting labels
- Train/test leakage
- Class distribution
- Text length
- Sample quality

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from datasets import load_dataset

print("Libraries imported successfully!")

Download BANKING77 directly from Hugging Face

In [ ]:
TRAIN_URL = (
    "https://raw.githubusercontent.com/"
    "PolyAI-LDN/task-specific-datasets/"
    "master/banking_data/train.csv"
)

TEST_URL = (
    "https://raw.githubusercontent.com/"
    "PolyAI-LDN/task-specific-datasets/"
    "master/banking_data/test.csv"
)

train_df = pd.read_csv(TRAIN_URL)
test_df = pd.read_csv(TEST_URL)

print("Training shape:", train_df.shape)
print("Testing shape:", test_df.shape)

train_df.head()

In [ ]:
train_df = train_df.rename(
    columns={"category": "intent"}
)

test_df = test_df.rename(
    columns={"category": "intent"}
)

train_df.head()

In [ ]:
intent_names = sorted(
    train_df["intent"].unique()
)

print("Number of intents:", len(intent_names))
print(intent_names[:20])

In [ ]:
print("Training examples:", len(train_df))
print("Testing examples:", len(test_df))
print("Number of intents:", train_df["intent"].nunique())

In [ ]:
print("TRAIN MISSING VALUES")
print(train_df.isnull().sum())

print("\nTEST MISSING VALUES")
print(test_df.isnull().sum())

In [ ]:
print(
    "Exact duplicate training texts:",
    train_df.duplicated(
        subset=["text"]
    ).sum()
)

print(
    "Exact duplicate test texts:",
    test_df.duplicated(
        subset=["text"]
    ).sum()
)

In [ ]:
train_label_counts = (
    train_df
    .groupby("text")["intent"]
    .nunique()
)

train_conflicts = train_label_counts[
    train_label_counts > 1
]

print(
    "Training texts with conflicting intents:",
    len(train_conflicts)
)

In [ ]:
train_texts = set(
    train_df["text"].str.strip()
)

test_texts = set(
    test_df["text"].str.strip()
)

overlap = train_texts.intersection(
    test_texts
)

print(
    "Exact texts appearing in both train and test:",
    len(overlap)
)

In [ ]:
overlap_examples = []

for text in sorted(overlap):
    train_labels = train_df.loc[
        train_df["text"].str.strip() == text,
        "intent"
    ].unique()

    test_labels = test_df.loc[
        test_df["text"].str.strip() == text,
        "intent"
    ].unique()

    overlap_examples.append({
        "text": text,
        "train_intent": list(train_labels),
        "test_intent": list(test_labels)
    })

overlap_df = pd.DataFrame(overlap_examples)

overlap_df

Strict de-duplicated test score

In [ ]:
strict_test_df = test_df[
    ~test_df["text"].str.strip().isin(train_texts)
].copy()

print("Official test size:", len(test_df))
print("Strict test size:", len(strict_test_df))

In [ ]:
strict_overlap = set(
    train_df["text"].str.strip()
).intersection(
    set(strict_test_df["text"].str.strip())
)

print(
    "Strict train/test exact overlap:",
    len(strict_overlap)
)

In [ ]:
def normalize_text(text):
    return " ".join(
        str(text)
        .lower()
        .strip()
        .split()
    )

train_normalized = set(
    train_df["text"].apply(normalize_text)
)

test_normalized = set(
    test_df["text"].apply(normalize_text)
)

normalized_overlap = train_normalized.intersection(
    test_normalized
)

print(
    "Normalized train/test overlap:",
    len(normalized_overlap)
)

In [ ]:
def normalize_text(text):
    return " ".join(
        str(text)
        .lower()
        .strip()
        .split()
    )


# Create normalized versions
train_df["normalized_text"] = train_df["text"].apply(normalize_text)
test_df["normalized_text"] = test_df["text"].apply(normalize_text)

# All normalized texts seen during training
train_normalized_texts = set(train_df["normalized_text"])

# Create strict test set
strict_test_df = test_df[
    ~test_df["normalized_text"].isin(train_normalized_texts)
].copy()

print("Official test size:", len(test_df))
print("Strict test size:", len(strict_test_df))

In [ ]:
strict_overlap = set(
    train_df["normalized_text"]
).intersection(
    set(strict_test_df["normalized_text"])
)

print(
    "Normalized overlap after strict filtering:",
    len(strict_overlap)
)

In [ ]:
overlap_records = []

for normalized_text in normalized_overlap:

    train_rows = train_df[
        train_df["normalized_text"] == normalized_text
    ]

    test_rows = test_df[
        test_df["normalized_text"] == normalized_text
    ]

    overlap_records.append({
        "text": normalized_text,
        "train_intents": train_rows["intent"].unique().tolist(),
        "test_intents": test_rows["intent"].unique().tolist()
    })

pd.DataFrame(overlap_records)

In [ ]:
strict_overlap = set(
    train_df["normalized_text"]
).intersection(
    set(strict_test_df["normalized_text"])
)

print(
    "Normalized overlap after strict filtering:",
    len(strict_overlap)
)